# 03 — Feature Engineering

Creates rolling-window statistics, lag features, rate-of-change, cross-sensor
interactions, and time-based features via `src/features/feature_engineering.py`.

## What this notebook does

1. Loads the preprocessed train / val / test splits.
2. Applies feature engineering and shows the resulting feature set.
3. Analyses feature correlation with the target.
4. Reports feature count before and after engineering.

**Prerequisite:** Run notebook 02 first.

## Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## 1 — Apply Feature Engineering

In [ ]:
from src.features.feature_engineering import engineer_features

for split in ("train", "val", "test"):
    df = pd.read_csv(PROCESSED_DIR / f"{split}.csv", parse_dates=["timestamp"])
    print(f"\n--- {split.upper()} ---")
    print(f"Before: {df.shape[1]} features")
    df_eng = engineer_features(df)
    print(f"After : {df_eng.shape[1]} features")
    df_eng.to_csv(PROCESSED_DIR / f"{split}_engineered.csv", index=False)
    print(f"Saved → {split}_engineered.csv")

## 2 — Inspect New Features

In [ ]:
df_train = pd.read_csv(PROCESSED_DIR / "train_engineered.csv", parse_dates=["timestamp"])
print("Columns:", list(df_train.columns))
df_train.head(3)

## 3 — Feature Correlation with Target (Top 20)

In [ ]:
drop_cols = ["timestamp", "machine_id", "time_to_failure"]
numeric = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns])
corr = numeric.corr()["failure_within_48h"].drop("failure_within_48h").abs().nlargest(20)

fig, ax = plt.subplots(figsize=(8, 7))
corr.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 20 Features — Correlation with failure_within_48h")
ax.set_xlabel("|Pearson r|")
plt.tight_layout()
plt.show()

## Next Steps

→ Open **04_baseline_models.ipynb** to train Random Forest and XGBoost.